# 실습 6 — Task 1. LangChain RAG 챗봇 (Day 3 / 60분)

> 시나리오: **사내 문서 기반 Q&A** — `data/sample_docs/` 의 회사 정책 문서
>
> 외부 예제 없이 LangChain + FAISS 로 RAG 파이프라인을 만든다.
> PDF 로 바꾸려면 `TextLoader` 대신 `PyMuPDFLoader("내문서.pdf")` 만 교체.

## RAG 5단계
**Load → Split → Embed → Retrieve → Generate**

## Task
1. 문서 적재 + 청크 분할
2. 임베딩 → FAISS 벡터스토어
3. Retriever + 프롬프트 + LCEL 체인 (**출처 없으면 '모름' 강제**)
4. 질문해 보기 → 멀티턴 RAG 로 확장 (TODO)

## 1) Load + Split

In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader(
    "../data/sample_docs", glob="**/*.md",
    loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()
print(f"문서 {len(docs)}개 로드")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"청크 {len(chunks)}개")

/tmp/ipykernel_4327/3222463839.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


문서 2개 로드
청크 4개


## 2) Embed → FAISS

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 검색 동작 확인
for d in retriever.invoke("연차 며칠 쓸 수 있어?"):
    print("-", d.page_content[:60].replace("\n", " "), "...")

- # (예시) 사내 인사 규정 — RAG 실습용 샘플 문서  > 실습 6(RAG)용 **가상의 회사 정책 문서 ...
- ## 경비 처리 - 업무 관련 경비는 **법인카드** 사용을 원칙으로 합니다. - 부득이하게 개인카드를 사용 ...
- # (예시) 사내 IT 보안 가이드 — RAG 실습용 샘플 문서  > 실습 6(RAG)용 **가상의 회사 정 ...


## 3) Retrieve + Generate (LCEL) — 출처 없으면 '모름'

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """아래 '문맥'만 근거로 한국어로 답하세요.
문맥에 답이 없으면 반드시 "문서에서 찾을 수 없습니다"라고만 답하세요.

[문맥]
{context}

[질문]
{question}
"""
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

## LCEL 뜯어보기 — `|` 를 따라 데이터가 흐른다

`rag_chain` 한 줄이 어렵게 느껴지면, **각 `|` 단계의 출력을 직접 찍어** 보면 된다.
파이프는 왼쪽 출력 → 오른쪽 입력으로 데이터를 흘려보낼 뿐이다.

```
"질문"
  ├─ context  : retriever | format_docs   (관련 청크 검색 → 한 덩어리 텍스트)
  └─ question : RunnablePassthrough()      (질문을 그대로 통과)
      ▼  = {"context": "...", "question": "질문"}
   prompt   (문맥+질문을 템플릿에 끼움)
      ▼
   llm      (LLM 호출 → 응답 객체)
      ▼
   StrOutputParser()   (응답 객체 → 순수 문자열)
```

즉 `rag_chain` 은 결국 **`setup | prompt | llm | StrOutputParser()`** 와 같다.

In [5]:
from langchain_core.runnables import RunnableParallel

q = "연차는 며칠이고 어떻게 신청해?"

# ① retriever | format_docs — 관련 청크를 검색해 한 덩어리 문자열로
ctx = (retriever | format_docs).invoke(q)
print("① context (검색+병합):\n", ctx[:200], "...\n")

# ② 두 갈래를 묶어 prompt 입력 dict 를 만든다 (rag_chain 의 {...} 부분)
setup = RunnableParallel({"context": retriever | format_docs,
                          "question": RunnablePassthrough()})
step2 = setup.invoke(q)
print("② prompt 입력 dict 의 키:", list(step2.keys()), "\n")

# ③ setup | prompt — 템플릿에 문맥/질문을 끼운 완성 프롬프트
filled = (setup | prompt).invoke(q)
print("③ 완성된 프롬프트(앞부분):\n", filled.to_string()[:300], "...\n")

# ④ 전체 체인 = setup | prompt | llm | StrOutputParser()
answer = (setup | prompt | llm | StrOutputParser()).invoke(q)
print("④ 최종 답변:\n", answer)

① context (검색+병합):
 # (예시) 사내 인사 규정 — RAG 실습용 샘플 문서

> 실습 6(RAG)용 **가상의 회사 정책 문서**입니다. 실제 규정이 아니며,
> 자유롭게 내 회사 문서로 교체해 다시 인덱싱해 보세요.

## 연차 휴가
- 입사 1년 차는 연 **15일**의 연차가 부여됩니다.
- 3년 이상 근속 시 2년마다 1일씩 가산되며, 최대 25일까지 늘어납니다.
- ...

② prompt 입력 dict 의 키: ['context', 'question'] 

③ 완성된 프롬프트(앞부분):
 Human: 아래 '문맥'만 근거로 한국어로 답하세요.
문맥에 답이 없으면 반드시 "문서에서 찾을 수 없습니다"라고만 답하세요.

[문맥]
# (예시) 사내 인사 규정 — RAG 실습용 샘플 문서

> 실습 6(RAG)용 **가상의 회사 정책 문서**입니다. 실제 규정이 아니며,
> 자유롭게 내 회사 문서로 교체해 다시 인덱싱해 보세요.

## 연차 휴가
- 입사 1년 차는 연 **15일**의 연차가 부여됩니다.
- 3년 이상 근속 시 2년마다 1일씩 가산되며, 최대 25일까지 늘어납니다.
- 연차는 사내 그룹웨어 **[근태] →  ...

④ 최종 답변:
 입사 1년 차는 연 **15일**의 연차가 부여되며, 연차는 사내 그룹웨어 **[근태] → [휴가 신청]** 메뉴에서 신청하며, 최소 **3일 전**에 등록해야 합니다.


## 4) 질문해 보기

In [4]:
print(rag_chain.invoke("연차는 며칠이고 어떻게 신청해?"))
print("---")
print(rag_chain.invoke("재택근무 규정 알려줘"))
print("---")
print(rag_chain.invoke("회사 주식 상장일이 언제야?"))   # 문서에 없음 → '찾을 수 없습니다'

입사 1년 차는 연 **15일**의 연차가 부여되며, 연차는 사내 그룹웨어 **[근태] → [휴가 신청]** 메뉴에서 신청하며, 최소 **3일 전**에 등록해야 합니다.
---
재택근무는 **주 2일**까지 가능하며, 팀장 승인을 받아야 합니다. 재택 시에도 코어타임 **10:00~16:00** 에는 메신저 응답이 가능해야 합니다. 신규 입사자는 온보딩 기간(첫 4주) 동안 재택근무가 제한됩니다.
---
문서에서 찾을 수 없습니다.


## 멀티턴 RAG 로 확장 (TODO)
- [ ] 이전 대화를 반영하도록 history-aware retriever 추가
      (질문을 history 로 *재작성* 후 검색)
- [ ] `data/sample_docs/` 에 내 회사 문서를 추가하고 다시 인덱싱
- [ ] PDF 문서로 교체: `from langchain_community.document_loaders import PyMuPDFLoader`

## 회고
- [ ] RAG 가 환각을 줄이는 지점은 어디인가? (출처 강제)